In [ ]:
import pandas as pd
import numpy as np

# preprocessing
import re
import string

# model + pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer

# models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
df = pd.read_csv('spam.csv')

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
df = df.rename(columns={'v1': 'label', 'v2': 'text'})
df = df[['label', 'text']]

In [ ]:
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [ ]:
df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = text.strip()  #remove extra space from beg. and end of string
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [ ]:
df.head()

,label,text,clean_text
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final...
3,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


In [ ]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y    #stratify=y ensures balanced spam/ham split
)


#stratify = it will split both train and test ham and spam in same proportion

In [ ]:
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))),
    ('model', LogisticRegression())
])

pipe_lr.fit(X_train, y_train)

y_pred_lr = pipe_lr.predict(X_test)

print("Logistic Regression Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Results:
Accuracy: 0.9650224215246637
[[965   1]
 [ 38 111]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       0.99      0.74      0.85       149

    accuracy                           0.97      1115
   macro avg       0.98      0.87      0.92      1115
weighted avg       0.97      0.97      0.96      1115



In [ ]:
word_tfidf = TfidfVectorizer(           #captures meaning
    stop_words='english',
    max_features=5000,
    ngram_range=(1,2),
    analyzer='word'
)

char_tfidf = TfidfVectorizer(           #captures patterns
    max_features=5000,
    ngram_range=(3,5),
    analyzer='char'
)

pipe_nb = Pipeline([
    ('features', FeatureUnion([
        ('word', word_tfidf),
        ('char', char_tfidf)
    ])),
    ('model', MultinomialNB())
])

pipe_nb.fit(X_train, y_train)

y_pred_nb = pipe_nb.predict(X_test)

print("Naive Bayes Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Results:
Accuracy: 0.9757847533632287
[[955  11]
 [ 16 133]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       966
           1       0.92      0.89      0.91       149

    accuracy                           0.98      1115
   macro avg       0.95      0.94      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
param_grid = {
    'features__word__max_features': [3000, 5000],
    'features__word__ngram_range': [(1,1), (1,2)],
    'features__char__ngram_range': [(3,5), (3,6)],
    'model__alpha': [0.5, 1.0]
}

grid = GridSearchCV(pipe_nb, param_grid, cv=3, scoring='f1', n_jobs=-1)   #Gridsearch CV with naive bayes

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

best_model = grid.best_estimator_

Best Params: {'features__char__ngram_range': (3, 5), 'features__word__max_features': 3000, 'features__word__ngram_range': (1, 1), 'model__alpha': 1.0}


In [ ]:
y_pred = best_model.predict(X_test)

print("Final Model Results:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Final Model Results:
Accuracy: 0.9739910313901345
[[951  15]
 [ 14 135]]
              precision    recall  f1-score   support

           0       0.99      0.98      0.98       966
           1       0.90      0.91      0.90       149

    accuracy                           0.97      1115
   macro avg       0.94      0.95      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [ ]:
messages = [
    "fr33 offer",
    "Hey, are we meeting today?"
]

preds = best_model.predict(messages)

for msg, pred in zip(messages, preds):
    print(msg, "->", "Spam" if pred == 1 else "Ham")

fr33 offer -> Spam
Hey, are we meeting today? -> Ham


In [ ]:
df['prediction'] = best_model.predict(df['clean_text'])

df.to_csv('spam_predictions.csv', index=False)

Precision = 0.90

When model says “spam”:
90% correct
10% false alarms


Recall = 0.91

Out of all actual spam:
91% detected
Only 9% missed